# Model Comparisson

### Requirements:

- **Lab 3 (due Mon Apr 20):** model comparison (≥3 families) + structured error analysis (failure-mode table) + one-page non-technical memo. Start thinking about your comparison strategy.


- This is the deliverable pattern. For every model you build from now on — Lab 3, Capstone, Final Exam — you will produce this table.

| Failure Mode | N (races) | F1 (slice) | F1 (overall) | Gap | Proposed Mitigation |
|---|---|---|---|---|---|
| Street circuits | ? | ? | ? | ? | ? |
| Wet races | ? | ? | ? | ? | ? |
| New circuits | ? | ? | ? | ? | ? |

> **⏸ Pause.** "This table turns a model evaluation into a robustness report. Lab 3 requires exactly this. Your Capstone requires exactly this. Learn the pattern now."


- EXECUTION > DESCRIPTION 	Your notebook must show visible outputs. A comparison table with real numbers from executed cells. If your notebook has no outputs when I open it, maximum methodology score is 50%. Run All before submitting.


- REASONING > METHODOLOGY 	A correct comparison table with no explanation of WHY each model performed the way it did scores lower than a table with fewer models but clear reasoning. Write reasoning in markdown cells. Explain your thinking in code comments.

- AUTHENTICITY > FORMAT 	PROMPTS.md is scored for traceability (can I trace your claims to your notebook?), operational detail (exact errors, specific metrics), and critical distance. Format compliance alone scores low.

"My model lost to the baseline" is a valid, high-scoring result — as long as you explain WHY it lost and what that tells you about the problem.

- ≥3 models + baseline compared. Same metric, same validation, same test set across ALL rows. Visible cell outputs in notebook. Train-vs-test gap reported for each model.

Key rule: Every row in your comparison table must use the same primary metric and the same temporal split. If they don't match, the comparison is invalid regardless of how many models you trained. The metric must be appropriate for your chosen framing (e.g., MAE for regression, macro F1 for classification).

New requirement: Include a Train [metric] column alongside your test metric. The gap between train and test performance is diagnostic — it tells you whether your model is overfitting.

- Notebook runs top-to-bottom in <10 min on fresh env. RANDOM_SEED = 414 throughout. Visible outputs for every cell. README with clear instructions. Environment specified.

## 0. Setup

In [ ]:
# ── Reproducibility Header ────────────────────────────────────────────
# Every notebook in IIT414W starts here. Do not skip this block.

import sys, random
import numpy as np
import warnings

RANDOM_SEED = 414  # Course constant. Do not change.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

# Environment check
print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Seed    : {RANDOM_SEED}')

# ── Dependency Guard ───────────────────────────────────────────────
# Ensures all required packages are installed in the active kernel.
# Safe to re-run: pip will skip already-installed packages.

import importlib, subprocess, sys

_REQUIRED = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'fastf1': 'fastf1',
}

_missing = []
for _mod, _pip in _REQUIRED.items():
    try:
        importlib.import_module(_mod)
    except ModuleNotFoundError:
        _missing.append(_pip)

if _missing:
    print(f'Installing missing packages: {_missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + _missing)
    print('Done. Packages installed successfully.')
else:
    print('All required packages already installed ✓')

# ── Library Imports ───────────────────────────────────────────────
import os                        # Working directory checks
import subprocess                # Git command checks
import importlib                 # Runtime dependency checks
import numpy as np               # Numeric support
import pandas as pd              # Tables and diagnostics
import matplotlib.pyplot as plt  # Plotting
import seaborn as sns            # Statistical plotting
import fastf1                    # Formula 1 data access

print(f'fastf1  : {fastf1.__version__}')
print(f'pandas  : {pd.__version__}')

In [ ]:
# Configure local cache so repeated API reads are fast and reproducible.
cache_path = os.path.abspath("fastf1_cache")
os.makedirs(cache_path, exist_ok=True)
fastf1.Cache.enable_cache(cache_path)

# Create Ergast interface and iterate all race weekends in each season.
erg = fastf1.ergast.Ergast()

years = [2019, 2020, 2021, 2022, 2023, 2024]
results_list = []

for year in years:
    schedule = fastf1.get_event_schedule(year)

    for _, event in schedule.iterrows():
        # Ignore testing sessions because they are not part of race prediction.
        if event['EventFormat'] == 'testing':
            continue

        rnd = int(event['RoundNumber'])

        try:
            res = erg.get_race_results(season=year, round=rnd)
            race = res.content[0]

            # Keep temporal/event metadata for later grouping and split logic.
            race['year'] = year
            race['round'] = rnd
            race['circuit'] = event['EventName']

            results_list.append(race)

        except Exception as e:
            print(f"Skipping {year} round {rnd}")

# Merge all race result frames into one modeling table.
df = pd.concat(results_list, ignore_index=True)

# Keep only columns used in this lab.
df = df[[
    'driverNumber',
    'givenName',
    'familyName',
    'constructorName',
    'grid',
    'position',
    'status',
    'points',
    'year',
    'round',
    'circuit'
]]

# Standardize column names for readability.
df.columns = [
    'DriverNumber',
    'FirstName',
    'LastName',
    'TeamName',
    'GridPosition',
    'Position',
    'Status',
    'Points',
    'Year',
    'Round',
    'Circuit'
]

# Build convenient identity and numeric fields.
df['FullName'] = df['FirstName'] + " " + df['LastName']
df['GridPosition'] = pd.to_numeric(df['GridPosition'], errors='coerce')
df['Position'] = pd.to_numeric(df['Position'], errors='coerce')

# Feature engineering for baseline and analysis.
df['Top10'] = ((df['Position'] <= 10) & (df['Position'] >0)).astype(int)
df['BaselinePred'] = (df['GridPosition'] <= 10).astype(int)
df['PositionsGained'] = df['GridPosition'] - df['Position']
df['DNF'] = ~df['Status'].str.contains("Finished", na=False)

# Historical reliability and constructor strength proxies.
driver_reliability = 1 - df.groupby('FullName')['DNF'].mean()
df['DriverReliability'] = df['FullName'].map(driver_reliability)
constructor_points = df.groupby('TeamName')['Points'].mean()
df['ConstructorStrength'] = df['TeamName'].map(constructor_points)

# Remove unusable rows for grid/finish-based analyses.
df = df.dropna(subset=['GridPosition','Position'])

print("Dataset shape:", df.shape)


>Our main target is going to be "DNF"

## Model 1: Logistic Regression

**Why?:** 

## Model 2: 

**Why?:** 

## Model 3: 

**Why?:** 

## Comparisson Table